# callbacks

> Focus change callback and audio playback JavaScript for the alignment card stack

In [ ]:
#| default_exp components.step_alignment.callbacks

In [ ]:
#| export
from cjm_fasthtml_card_stack.core.config import CardStackConfig
from cjm_fasthtml_card_stack.core.html_ids import CardStackHtmlIds
from cjm_fasthtml_card_stack.core.button_ids import CardStackButtonIds
from cjm_fasthtml_card_stack.core.models import CardStackUrls
from cjm_fasthtml_card_stack.js.core import generate_card_stack_js

## Focus Change + Audio Audition

When the user navigates to a VAD chunk, the focus change callback:
1. Updates the hidden input with the focused chunk index
2. Auto-plays the audio for the chunk's time range (audition pattern)

Audio playback is robust to missing audio elements or source — it silently no-ops.

In [ ]:
#| export
def _generate_align_focus_change_script(
    focus_input_id:str,  # ID of hidden input for focused chunk index
    audio_player_id:str,  # ID of the hidden audio element
) -> str:  # JavaScript for focus change with audio playback and playing indicator
    """Generate JS for VAD chunk focus change handling with audio audition and visual feedback."""
    return f"""
        // Called when card focus changes in the alignment zone
        window.onAlignFocusChange = function(item, index, zoneId) {{
            // Update hidden input with focused chunk index
            var input = document.getElementById('{focus_input_id}');
            if (input && item) {{
                input.value = item.dataset.chunkIndex || index;
            }}

            // Hide all playing indicators
            document.querySelectorAll('.vad-playing-indicator').forEach(function(el) {{
                el.classList.add('invisible');
                el.classList.remove('visible');
            }});

            // Audio audition — play the chunk's time range
            var player = document.getElementById('{audio_player_id}');
            if (!player || !player.src || !item) return;

            var start = parseFloat(item.dataset.startTime || 0);
            var end = parseFloat(item.dataset.endTime || 0);
            if (end <= start) return;

            // Cancel previous playback
            if (window._alignPlayTimeout) clearTimeout(window._alignPlayTimeout);
            player.pause();

            // Show playing indicator on current card
            var indicator = item.querySelector('.vad-playing-indicator');
            if (indicator) {{
                indicator.classList.remove('invisible');
                indicator.classList.add('visible');
            }}

            // Play chunk time range
            player.currentTime = start;
            player.play().catch(function() {{}});
            window._alignPlayTimeout = setTimeout(function() {{
                player.pause();
                // Hide indicator when playback ends
                if (indicator) {{
                    indicator.classList.add('invisible');
                    indicator.classList.remove('visible');
                }}
            }}, (end - start) * 1000);
        }};
    """

## Combined Callbacks Script

Wraps the card stack library's JS generator with alignment-specific focus change callback.

In [ ]:
#| export
def generate_align_callbacks_script(
    ids:CardStackHtmlIds,  # Card stack HTML IDs
    button_ids:CardStackButtonIds,  # Card stack button IDs
    config:CardStackConfig,  # Card stack configuration
    urls:CardStackUrls,  # Card stack URL bundle
    container_id:str,  # ID of the alignment container (parent of card stack)
    focus_input_id:str,  # ID of hidden input for focused chunk index
    audio_player_id:str,  # ID of the hidden audio element
) -> any:  # Script element with all JavaScript callbacks
    """Generate JavaScript for alignment card stack with audio audition."""
    align_scripts = (
        _generate_align_focus_change_script(focus_input_id, audio_player_id),
    )

    return generate_card_stack_js(
        ids=ids,
        button_ids=button_ids,
        config=config,
        urls=urls,
        container_id=container_id,
        extra_scripts=align_scripts
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()